In [1]:
import requests
from requests.packages.urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import csv

In [2]:
class TimeoutHttpAdapter(HTTPAdapter):
    def __init__(self, timeout=None, *args, **kwargs):
        self.timeout = timeout
        if "timeout" in kwargs:
            del kwargs["timeout"]
        super().__init__(*args, **kwargs)

    def send(self, *args, **kwargs):
        kwargs['timeout'] = self.timeout
        return super().send(*args, **kwargs)

In [3]:
r=requests.Session()
retry_strategy = Retry(
            total=3,
            status_forcelist=[104, 429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET" "POST", "PUT", "DELETE", "OPTIONS", "TRACE"],
            backoff_factor=2
        )
r.headers["User-Agent"]='My User Agent 1.0'
r.mount('https://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))
r.mount('http://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))

req = r.get('https://www.lawyersclubindia.com/articles/default.asp?cat_id=7&member_id=&popular=&offset=1')

In [17]:
rows=[]
for page in range(1,9):
    lnk='https://www.lawyersclubindia.com/articles/default.asp?cat_id=7&member_id=&popular=&offset='+str(page)
    req = r.get(lnk)
    soup = BeautifulSoup(req.content, 'html.parser')
    s = soup.find_all('a', class_='text-dark semi-bold')
    link_list=[]
    for a in s:
        link_list.append('https://www.lawyersclubindia.com/articles/'+a.attrs["href"])
    for link in link_list:
        req=r.get(link)
        soup = BeautifulSoup(req.content, "html.parser")
        title=soup.find('a',class_='text-dark').text
        content=soup.find('div',class_='ft-page-content dont-break-out').text
        rows.append([title,content])

In [22]:
fields = ['title', 'content'] 
with open('legal_articles_lawyersclub_criminal_law.csv', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)
f.close()